In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sys.path.append('notebooks/src')
from Data_Loader import load_cleaned     # ← uses cleaned files now

sales, channels, products, customers, delivery = load_cleaned()
os.makedirs(r"C:\Users\Hello\Downloads\project\BoatSalesProject\data\outputs", exist_ok=True)
print("✅ Cleaned data loaded!")

In [ ]:
#monthly revenue trend
monthly = sales.groupby(['Year', 'Month'])['NetSales'].sum().reset_index()

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly, x='Month', y='NetSales', hue='Year', marker='o')
plt.title('Monthly Revenue Trend')
plt.tight_layout()
plt.savefig(r"C:\Users\Hello\Downloads\project\BoatSalesProject\data\outputs\monthly_revenue.png")
plt.show()
print("✅ Chart saved")

In [ ]:
#top 10 product
top_products = (sales.groupby('ProductID')['NetSales']
                .sum().sort_values(ascending=False).head(10))

top_products.plot(kind='bar', figsize=(12, 5), color='steelblue')
plt.title('Top 10 Products by Revenue')
plt.tight_layout()
plt.savefig(r"C:\Users\Hello\Downloads\project\BoatSalesProject\data\outputs\top_products.png")
plt.show()

In [ ]:
#channel performance
channel_perf = (sales.groupby('ChannelType')
                .agg(TotalRevenue=('NetSales','sum'),
                     TotalProfit=('Profit','sum'),
                     OrderCount=('SalesID','count'))
                .reset_index())

channel_perf['ProfitMargin'] = (channel_perf['TotalProfit'] /
                                channel_perf['TotalRevenue'] * 100).round(2)
print(channel_perf)

In [ ]:
#customer tier analysis
tier_analysis = (sales.groupby('CustomerTier')
                 .agg(AvgOrderValue=('NetSales','mean'),
                      TotalOrders=('SalesID','count'),
                      TotalRevenue=('NetSales','sum'))
                 .reset_index())
print(tier_analysis)

In [ ]:
fest = sales.groupby('IsFestivalPeriod')['NetSales'].agg(['mean','sum','count'])
print(fest)

fest['sum'].plot(kind='bar', color=['#2196F3','#FF9800'])
plt.title('Total Sales: Festival vs Non-Festival')
plt.xticks([0,1], ['Non-Festival','Festival'], rotation=0)
plt.tight_layout()
plt.savefig(r"C:\Users\Hello\Downloads\project\BoatSalesProject\data\outputs\festival_sales.png")
plt.show()

In [ ]:
status_counts = sales['OrderStatus'].value_counts()
status_counts.plot(kind='pie', autopct='%1.1f%%', figsize=(7, 7))
plt.title('Order Status Distribution')
plt.savefig(r'C:\Users\Hello\Downloads\project\BoatSalesProject\data\outputs\order_status.png')
plt.show()

cancelled = sales[sales['OrderStatus'] == 'Cancelled']
cancelled['CancellationReason'].value_counts().plot(kind='barh')
plt.title('Top Cancellation Reasons')
plt.tight_layout()
plt.savefig(r'C:\Users\Hello\Downloads\project\BoatSalesProject\data\outputs\cancellation_reasons.png')
plt.show()

In [ ]:
merged_del = sales.merge(delivery, on='DeliveryPartnerID')

del_perf = (merged_del.groupby('PartnerName')
            .agg(AvgDeliveryDays=('DeliveryDays','mean'),
                 TotalOrders=('SalesID','count'),
                 AvgRating=('Rating','mean'))
            .sort_values('AvgDeliveryDays')
            .reset_index())
print(del_perf)